In [0]:
import pyspark.sql.functions
import requests
import json

#### Reading files

In [0]:
my_volume_dir = "/Volumes/dbr_dev/lechster10/databricks_course/first_week_resources"

path_game = f"{my_volume_dir}/game.json"
path_game_summ = f"{my_volume_dir}/game_summary.csv"
path_team = f"{my_volume_dir}/team.csv"

df_game = spark.read.option("multiLine", "true").format("json").load(path_game)
df_game_summ = spark.read.format("csv").option("header","true").load(path_game_summ)
df_team = spark.read.format("csv").option("header","true").load(path_team)


#### Loading into delta table

In [0]:
catalog = "dbr_dev"
schema = "lechster10"

In [0]:
df_game.write.format("delta").saveAsTable(f"{catalog}.{schema}.game_table")
df_game_summ.write.format("delta").saveAsTable(f"{catalog}.{schema}.df_game_summ_table")
df_team.write.format("delta").saveAsTable(f"{catalog}.{schema}.df_team_table")


#### Simple operations

In [0]:
df_game.show(3)

In [0]:
df_game_summ.show(3)

In [0]:
df_team.show(3)

In [0]:
old_column = "full_name"

results_df = df_game_summ.join(df_game, "game_id")
results_with_team_names = results_df.join(df_team, results_df["home_team_id"] == df_team["id"])   # adding home team name
results_with_team_names = results_with_team_names.select(["game_id", "game_date_est", "visitor_team_id", "season", "pts_home", "pts_away", old_column]) 
results_with_team_names = results_with_team_names.withColumnRenamed('full_name', 'home_team_name')

results_with_team_names = results_with_team_names.join(df_team, results_df["visitor_team_id"] == df_team["id"])   # adding visitor team name
results_with_team_names = results_with_team_names.select(["game_id", "game_date_est", "visitor_team_id", "season", "pts_home", "pts_away", "home_team_name", old_column]) 
results_with_team_names = results_with_team_names.withColumnRenamed('full_name', 'visitor_team_name')

results_with_team_names.show()


In [0]:
results_with_team_names.write.format("delta").saveAsTable(f"{catalog}.{schema}.results_with_team_names_table")

#### Most points in 2022 season

In [0]:
total_points_home = results_with_team_names.filter(results_with_team_names.season == "2022").groupby(results_with_team_names.home_team_name).agg({"pts_home": "sum"})
total_points_away = results_with_team_names.filter(results_with_team_names.season == "2022").groupby(results_with_team_names.visitor_team_name).agg({"pts_away": "sum"})

total_points_home = total_points_home.withColumnRenamed("sum(pts_home)", "sum_points_home")
total_points_away = total_points_away.withColumnRenamed("sum(pts_away)", "sum_points_away")

aggregated_points = total_points_home.join(total_points_away, total_points_home.home_team_name == total_points_away.visitor_team_name)
total_points = aggregated_points.select(aggregated_points.home_team_name.alias("team_name"), (aggregated_points.sum_points_home + aggregated_points.sum_points_away).alias('total'))

total_points.orderBy(total_points.total.desc()).show()

#### Loading data from an external API
##### getting 25 NBA random players 

In [0]:
SECRET_SCOPE = "default2" 
API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="blech-api-key")
url = "https://api.balldontlie.io/v1/players"

headers = { "Authorization": API_KEY }

response = requests.get(url, headers = headers)

if response.status_code == 200:
    print("Request was successful")
    data = response.json()
    list_of_players = data["data"]
else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")



In [0]:
list_of_players

In [0]:
path_json = f"{my_volume_dir}/players.json"

with open(path_json, "w") as f:
    json.dump(list_of_players, f)

#### Delta lake benefits

- ACID protects us from the situation when the transaction is unfinished. In this situation log is not created so even though we will have additional parquets files on disk. The system will not see them, because logs wont give it appropriate info.
- Time travel is a great benefit, because we can get a table state which was some time ago. It protects us from different accidents for instance overwriting our tables.
- Schema enforcement supervises table structure. It makes no trash go into our table (for instance it supervises column types and raises errors if needed)